In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ssrijanreddy/crashdataset/Motor_Vehicle_Collisions_-_Crashes_20260427.csv


In [5]:
# ============================================================
# CELL 1: BUILD MEMORY-SAFE ML DATASET FROM CRASH CSV
# Kaggle Version - SafeRoute AI
# ============================================================

!pip install -q pandas numpy scikit-learn folium branca joblib pyarrow matplotlib

import os
import gc
import pandas as pd
import numpy as np

WORK_DIR = "/kaggle/working"
INPUT_DIR = "/kaggle/input"

# -----------------------------
# 1. Find uploaded CSV inside /kaggle/input
# -----------------------------
csv_files = []

for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(".csv"):
            full_path = os.path.join(root, file)
            csv_files.append(full_path)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found in /kaggle/input. Please attach your dataset to the notebook.")

print("CSV files found:")
for f in csv_files:
    print(f, "size MB:", round(os.path.getsize(f) / (1024 * 1024), 2))

# Pick largest CSV as original crash dataset
file_path = max(csv_files, key=lambda f: os.path.getsize(f))
print("\nUsing crash dataset:", file_path)


# -----------------------------
# 2. Normalize column names
# -----------------------------
def normalize_col(col):
    return str(col).upper().replace("_", " ").strip()


# -----------------------------
# 3. Load only needed columns
# -----------------------------
wanted_columns = {
    "CRASH DATE",
    "CRASH TIME",
    "BOROUGH",
    "LATITUDE",
    "LONGITUDE",
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED"
}

all_columns = pd.read_csv(file_path, nrows=0).columns.tolist()

usecols = [
    col for col in all_columns
    if normalize_col(col) in wanted_columns
]

print("\nColumns being loaded:")
print(usecols)

df = pd.read_csv(
    file_path,
    usecols=usecols,
    low_memory=False
)

df.columns = [normalize_col(col) for col in df.columns]

print("\nRaw loaded shape:", df.shape)
display(df.head())


# -----------------------------
# 4. Basic cleaning
# -----------------------------
required_cols = ["CRASH DATE", "CRASH TIME", "LATITUDE", "LONGITUDE"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df.dropna(subset=["CRASH DATE", "CRASH TIME", "LATITUDE", "LONGITUDE"])

df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")

df = df.dropna(subset=["LATITUDE", "LONGITUDE"])

df = df[
    (df["LATITUDE"] != 0) &
    (df["LONGITUDE"] != 0)
]

# NYC coordinate filter
df = df[
    (df["LATITUDE"].between(40.3, 41.0)) &
    (df["LONGITUDE"].between(-74.3, -73.6))
]

if "NUMBER OF PERSONS INJURED" not in df.columns:
    df["NUMBER OF PERSONS INJURED"] = 0

if "NUMBER OF PERSONS KILLED" not in df.columns:
    df["NUMBER OF PERSONS KILLED"] = 0

df["NUMBER OF PERSONS INJURED"] = pd.to_numeric(
    df["NUMBER OF PERSONS INJURED"],
    errors="coerce"
).fillna(0).astype("float32")

df["NUMBER OF PERSONS KILLED"] = pd.to_numeric(
    df["NUMBER OF PERSONS KILLED"],
    errors="coerce"
).fillna(0).astype("float32")

if "BOROUGH" not in df.columns:
    df["BOROUGH"] = "UNKNOWN"

df["BOROUGH"] = df["BOROUGH"].fillna("UNKNOWN").astype(str)

df["CRASH DATETIME"] = pd.to_datetime(
    df["CRASH DATE"].astype(str) + " " + df["CRASH TIME"].astype(str),
    errors="coerce"
)

df = df.dropna(subset=["CRASH DATETIME"])

print("\nAfter cleaning:", df.shape)
display(df.head())


# -----------------------------
# 5. Create grid zones
# -----------------------------
GRID_ROUNDING = 3

df["lat_grid"] = df["LATITUDE"].round(GRID_ROUNDING).astype("float32")
df["lon_grid"] = df["LONGITUDE"].round(GRID_ROUNDING).astype("float32")
df["datetime_hour"] = df["CRASH DATETIME"].dt.floor("h")

grid_lookup = df[["lat_grid", "lon_grid", "BOROUGH"]].drop_duplicates().reset_index(drop=True)
grid_lookup["grid_code"] = np.arange(len(grid_lookup), dtype=np.int32)

df = df.merge(
    grid_lookup,
    on=["lat_grid", "lon_grid", "BOROUGH"],
    how="left"
)

print("\nUnique grid zones:", df["grid_code"].nunique())


# -----------------------------
# 6. Positive samples
# -----------------------------
positive_data = df.groupby(
    ["grid_code", "lat_grid", "lon_grid", "BOROUGH", "datetime_hour"],
    observed=True
).agg(
    accident_count=("grid_code", "size"),
    injured_count=("NUMBER OF PERSONS INJURED", "sum"),
    killed_count=("NUMBER OF PERSONS KILLED", "sum")
).reset_index()

positive_data["label"] = 1

positive_data["grid_code"] = positive_data["grid_code"].astype("int32")
positive_data["lat_grid"] = positive_data["lat_grid"].astype("float32")
positive_data["lon_grid"] = positive_data["lon_grid"].astype("float32")
positive_data["accident_count"] = positive_data["accident_count"].astype("int16")
positive_data["injured_count"] = positive_data["injured_count"].astype("float32")
positive_data["killed_count"] = positive_data["killed_count"].astype("float32")
positive_data["label"] = positive_data["label"].astype("int8")

print("\nPositive samples:", positive_data.shape)
display(positive_data.head())


# -----------------------------
# 7. Negative samples
# -----------------------------
unique_grids = positive_data[
    ["grid_code", "lat_grid", "lon_grid", "BOROUGH"]
].drop_duplicates().reset_index(drop=True)

all_hours = pd.date_range(
    positive_data["datetime_hour"].min(),
    positive_data["datetime_hour"].max(),
    freq="h"
)

hour_lookup = pd.DataFrame({
    "datetime_hour": all_hours,
    "hour_code": np.arange(len(all_hours), dtype=np.int32)
})

positive_temp = positive_data[["grid_code", "datetime_hour"]].merge(
    hour_lookup,
    on="datetime_hour",
    how="left"
)

n_hours = len(all_hours)

positive_keys = (
    positive_temp["grid_code"].astype(np.int64).values * n_hours
    + positive_temp["hour_code"].astype(np.int64).values
)

positive_key_set = set(positive_keys)

NEGATIVE_RATIO = 1
target_negatives = len(positive_data) * NEGATIVE_RATIO

rng = np.random.default_rng(42)

negative_parts = []
batch_size = 300000

while sum(len(x) for x in negative_parts) < target_negatives:
    sampled_grid_idx = rng.integers(0, len(unique_grids), size=batch_size)
    sampled_hour_code = rng.integers(0, n_hours, size=batch_size)

    sampled_grid_codes = unique_grids.iloc[sampled_grid_idx]["grid_code"].values.astype(np.int32)

    sampled_keys = sampled_grid_codes.astype(np.int64) * n_hours + sampled_hour_code.astype(np.int64)

    mask = np.fromiter(
        (key not in positive_key_set for key in sampled_keys),
        dtype=bool,
        count=len(sampled_keys)
    )

    temp = pd.DataFrame({
        "grid_code": sampled_grid_codes[mask].astype(np.int32),
        "hour_code": sampled_hour_code[mask].astype(np.int32)
    })

    temp = temp.drop_duplicates()

    negative_parts.append(temp)

    current_count = sum(len(x) for x in negative_parts)
    print("Negative samples collected:", current_count)

negative_codes = pd.concat(negative_parts, ignore_index=True).drop_duplicates()
negative_codes = negative_codes.head(target_negatives)

negative_data = negative_codes.merge(
    unique_grids,
    on="grid_code",
    how="left"
)

negative_data = negative_data.merge(
    hour_lookup,
    on="hour_code",
    how="left"
)

negative_data = negative_data.drop(columns=["hour_code"])

negative_data["accident_count"] = 0
negative_data["injured_count"] = 0
negative_data["killed_count"] = 0
negative_data["label"] = 0

negative_data["grid_code"] = negative_data["grid_code"].astype("int32")
negative_data["lat_grid"] = negative_data["lat_grid"].astype("float32")
negative_data["lon_grid"] = negative_data["lon_grid"].astype("float32")
negative_data["accident_count"] = negative_data["accident_count"].astype("int16")
negative_data["injured_count"] = negative_data["injured_count"].astype("float32")
negative_data["killed_count"] = negative_data["killed_count"].astype("float32")
negative_data["label"] = negative_data["label"].astype("int8")

print("\nNegative samples:", negative_data.shape)
display(negative_data.head())


# -----------------------------
# 8. Combine positive + negative
# -----------------------------
ml_data = pd.concat(
    [positive_data, negative_data],
    ignore_index=True
)

ml_data = ml_data.sample(frac=1, random_state=42).reset_index(drop=True)

ml_data["datetime_hour"] = pd.to_datetime(ml_data["datetime_hour"])

ml_data = ml_data.rename(columns={"BOROUGH": "borough"})
ml_data["borough"] = ml_data["borough"].astype(str).fillna("UNKNOWN")


# -----------------------------
# 9. Time features
# -----------------------------
ml_data["year"] = ml_data["datetime_hour"].dt.year.astype("int16")
ml_data["month"] = ml_data["datetime_hour"].dt.month.astype("int8")
ml_data["hour"] = ml_data["datetime_hour"].dt.hour.astype("int8")
ml_data["day_of_week"] = ml_data["datetime_hour"].dt.dayofweek.astype("int8")
ml_data["is_weekend"] = ml_data["day_of_week"].isin([5, 6]).astype("int8")
ml_data["is_night"] = ((ml_data["hour"] >= 20) | (ml_data["hour"] <= 5)).astype("int8")


# -----------------------------
# 10. Past accident history
# -----------------------------
cutoff_time = ml_data["datetime_hour"].quantile(0.8)

train_part_for_history = ml_data[
    (ml_data["datetime_hour"] < cutoff_time) &
    (ml_data["label"] == 1)
].copy()

history = train_part_for_history.groupby("grid_code").agg(
    past_accidents=("accident_count", "sum"),
    past_injuries=("injured_count", "sum"),
    past_deaths=("killed_count", "sum")
).reset_index()

ml_data = ml_data.merge(
    history,
    on="grid_code",
    how="left"
)

history_cols = ["past_accidents", "past_injuries", "past_deaths"]

for col in history_cols:
    ml_data[col] = ml_data[col].fillna(0).astype("float32")


# -----------------------------
# 11. Save checkpoint
# -----------------------------
checkpoint_path = os.path.join(WORK_DIR, "ml_data_checkpoint.parquet")

ml_data.to_parquet(checkpoint_path, index=False)

print("\n✅ ML dataset created and saved.")
print("Saved file:", checkpoint_path)
print("ML data shape:", ml_data.shape)
print("\nLabel counts:")
print(ml_data["label"].value_counts())

display(ml_data.head())


# -----------------------------
# 12. Free RAM
# -----------------------------
del df
del positive_temp
del negative_parts
del negative_codes

gc.collect()

print("\nRAM cleanup done. Continue to Cell 2.")

CSV files found:
/kaggle/input/datasets/ssrijanreddy/crashdataset/Motor_Vehicle_Collisions_-_Crashes_20260427.csv size MB: 538.34

Using crash dataset: /kaggle/input/datasets/ssrijanreddy/crashdataset/Motor_Vehicle_Collisions_-_Crashes_20260427.csv

Columns being loaded:
['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'LATITUDE', 'LONGITUDE', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED']

Raw loaded shape: (2257129, 7)


,CRASH DATE,CRASH TIME,BOROUGH,LATITUDE,LONGITUDE,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED
0,09/11/2021,2:39,NaN,NaN,NaN,2.0,0.0
1,03/26/2022,11:45,NaN,NaN,NaN,1.0,0.0
2,11/01/2023,1:29,BROOKLYN,40.62179,-73.970024,1.0,0.0
3,06/29/2022,6:55,NaN,NaN,NaN,0.0,0.0
4,09/21/2022,13:21,NaN,NaN,NaN,0.0,0.0



After cleaning: (2008927, 8)


,CRASH DATE,CRASH TIME,BOROUGH,LATITUDE,LONGITUDE,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,CRASH DATETIME
2,11/01/2023,1:29,BROOKLYN,40.621790,-73.970024,1.0,0.0,2023-11-01 01:29:00
9,09/11/2021,9:35,BROOKLYN,40.667202,-73.866500,0.0,0.0,2021-09-11 09:35:00
10,12/14/2021,8:13,BROOKLYN,40.683304,-73.917274,0.0,0.0,2021-12-14 08:13:00
12,12/14/2021,17:05,UNKNOWN,40.709183,-73.956825,0.0,0.0,2021-12-14 17:05:00
13,12/14/2021,8:17,BRONX,40.868160,-73.831480,2.0,0.0,2021-12-14 08:17:00



Unique grid zones: 82547

Positive samples: (1997445, 9)


...

Negative samples collected: 299944
Negative samples collected: 599888
Negative samples collected: 899839
Negative samples collected: 1199771
Negative samples collected: 1499705
Negative samples collected: 1799645
Negative samples collected: 2099563

Negative samples: (1997445, 9)


,grid_code,lat_grid,lon_grid,BOROUGH,datetime_hour,accident_count,injured_count,killed_count,label
0,7367,40.601002,-74.065002,STATEN ISLAND,2015-08-26 01:00:00,0,0.0,0.0,0
1,63887,40.702000,-73.959999,BROOKLYN,2020-10-16 20:00:00,0,0.0,0.0,0
2,54032,40.866001,-73.839996,BRONX,2026-03-10 11:00:00,0,0.0,0.0,0
3,36228,40.632999,-73.899002,UNKNOWN,2022-08-18 06:00:00,0,0.0,0.0,0
4,35744,40.682999,-73.919998,UNKNOWN,2015-11-11 11:00:00,0,0.0,0.0,0



✅ ML dataset created and saved.
Saved file: /kaggle/working/ml_data_checkpoint.parquet
ML data shape: (3994890, 18)

Label counts:
label
0    1997445
1    1997445
Name: count, dtype: int64


,grid_code,lat_grid,lon_grid,borough,datetime_hour,accident_count,injured_count,killed_count,label,year,month,hour,day_of_week,is_weekend,is_night,past_accidents,past_injuries,past_deaths
0,38018,40.613998,-74.101997,STATEN ISLAND,2019-08-14 04:00:00,0,0.0,0.0,0,2019,8,4,2,0,1,2.0,0.0,0.0
1,67828,40.728001,-73.738998,UNKNOWN,2022-02-20 11:00:00,0,0.0,0.0,0,2022,2,11,6,1,0,2.0,0.0,0.0
2,48110,40.577999,-73.848000,QUEENS,2019-02-13 21:00:00,0,0.0,0.0,0,2019,2,21,2,0,1,7.0,2.0,0.0
3,33179,40.646999,-74.023003,BROOKLYN,2018-02-06 06:00:00,1,1.0,0.0,1,2018,2,6,1,0,0,84.0,6.0,0.0
4,16307,40.636002,-73.984001,BROOKLYN,2026-01-06 19:00:00,1,1.0,0.0,1,2026,1,19,1,0,0,79.0,21.0,0.0



RAM cleanup done. Continue to Cell 2.


In [6]:
# ============================================================
# CELL 2: ADD WEATHER + ROAD STRUCTURE, TRAIN MODEL, FINAL OUTPUT
# Kaggle Version - SafeRoute AI
# ============================================================

!pip install -q requests osmnx geopandas shapely scipy lightgbm folium branca joblib matplotlib pyarrow

import os
import gc
import time
import warnings
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

WORK_DIR = "/kaggle/working"


# ============================================================
# 1. Load checkpoint
# ============================================================
checkpoint_path = os.path.join(WORK_DIR, "ml_data_checkpoint.parquet")

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError("ml_data_checkpoint.parquet not found. Run Cell 1 first.")

ml_data = pd.read_parquet(checkpoint_path)
ml_data["datetime_hour"] = pd.to_datetime(ml_data["datetime_hour"])

print("Loaded ml_data:", ml_data.shape)
display(ml_data.head())


# ============================================================
# 2. Add weather features from Open-Meteo
# ============================================================
NYC_LAT = 40.7128
NYC_LON = -74.0060

start_date = pd.to_datetime(ml_data["datetime_hour"].min()).date()
end_date = pd.to_datetime(ml_data["datetime_hour"].max()).date()

print("Fetching historical weather from", start_date, "to", end_date)

weather_parts = []

current_start = pd.Timestamp(start_date)
final_end = pd.Timestamp(end_date)

while current_start <= final_end:
    current_end = min(
        current_start + pd.DateOffset(years=1) - pd.Timedelta(days=1),
        final_end
    )

    weather_url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={NYC_LAT}&longitude={NYC_LON}"
        f"&start_date={current_start.date()}&end_date={current_end.date()}"
        "&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,"
        "weather_code,wind_speed_10m,surface_pressure"
        "&timezone=America%2FNew_York"
    )

    response = requests.get(weather_url, timeout=120)
    data = response.json()

    if "hourly" not in data:
        print(data)
        raise ValueError("Weather API failed. Make sure Kaggle Internet is ON.")

    part = pd.DataFrame(data["hourly"])
    part["datetime_hour"] = pd.to_datetime(part["time"]).dt.floor("h")
    part = part.drop(columns=["time"])

    weather_parts.append(part)

    print("Fetched:", current_start.date(), "to", current_end.date())

    current_start = current_end + pd.Timedelta(days=1)
    time.sleep(0.2)

weather_df = pd.concat(weather_parts, ignore_index=True)
weather_df = weather_df.drop_duplicates("datetime_hour")

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "wind_speed_10m",
    "surface_pressure"
]

for col in weather_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce")

print("Weather data:", weather_df.shape)
display(weather_df.head())

ml_data = ml_data.merge(
    weather_df,
    on="datetime_hour",
    how="left"
)

for col in weather_cols:
    ml_data[col] = pd.to_numeric(ml_data[col], errors="coerce")
    ml_data[col] = ml_data[col].fillna(ml_data[col].median()).astype("float32")

ml_data["is_rainy"] = (ml_data["rain"] > 0).astype("int8")
ml_data["is_heavy_rain"] = (ml_data["rain"] > 2).astype("int8")
ml_data["is_windy"] = (
    ml_data["wind_speed_10m"] > ml_data["wind_speed_10m"].quantile(0.75)
).astype("int8")

print("✅ Weather features added.")


# ============================================================
# 3. Add road structure and road type from OpenStreetMap
# ============================================================
import osmnx as ox
import geopandas as gpd
from scipy.spatial import cKDTree

grid_points = ml_data[
    ["grid_code", "lat_grid", "lon_grid", "borough"]
].drop_duplicates().reset_index(drop=True)

left = float(grid_points["lon_grid"].min() - 0.02)
right = float(grid_points["lon_grid"].max() + 0.02)
bottom = float(grid_points["lat_grid"].min() - 0.02)
top = float(grid_points["lat_grid"].max() + 0.02)

bbox = (left, bottom, right, top)

print("Downloading OSM road network for bbox:")
print("left, bottom, right, top =", bbox)

try:
    G = ox.graph_from_bbox(
        bbox=bbox,
        network_type="drive",
        simplify=True
    )
except TypeError:
    G = ox.graph_from_bbox(
        top,
        bottom,
        right,
        left,
        network_type="drive",
        simplify=True
    )

nodes, edges = ox.graph_to_gdfs(G)

print("OSM nodes:", nodes.shape)
print("OSM edges:", edges.shape)

nodes_m = nodes.to_crs(epsg=3857)
edges_m = edges.to_crs(epsg=3857)

grid_gdf = gpd.GeoDataFrame(
    grid_points,
    geometry=gpd.points_from_xy(grid_points["lon_grid"], grid_points["lat_grid"]),
    crs="EPSG:4326"
)

grid_m = grid_gdf.to_crs(epsg=3857)

if "length" not in edges_m.columns:
    edges_m["length"] = edges_m.geometry.length

edges_work = edges_m.reset_index().copy()
edges_work["midpoint"] = edges_work.geometry.interpolate(0.5, normalized=True)
edges_work["mid_x"] = edges_work["midpoint"].x
edges_work["mid_y"] = edges_work["midpoint"].y

edge_coords = np.vstack([
    edges_work["mid_x"].values,
    edges_work["mid_y"].values
]).T

edge_tree = cKDTree(edge_coords)

grid_coords = np.vstack([
    grid_m.geometry.x.values,
    grid_m.geometry.y.values
]).T

dist_to_edge, nearest_edge_idx = edge_tree.query(grid_coords, k=1)

BUFFER_METERS = 300
nearby_edge_indices = edge_tree.query_ball_point(grid_coords, r=BUFFER_METERS)

nodes_work = nodes_m.reset_index().copy()

if "street_count" in nodes_work.columns:
    intersection_nodes = nodes_work[nodes_work["street_count"] >= 3].copy()
else:
    intersection_nodes = nodes_work.copy()

node_coords = np.vstack([
    intersection_nodes.geometry.x.values,
    intersection_nodes.geometry.y.values
]).T

node_tree = cKDTree(node_coords)

nearby_node_indices = node_tree.query_ball_point(grid_coords, r=BUFFER_METERS)

road_rows = []

for i in range(len(grid_points)):
    nearest_idx = nearest_edge_idx[i]
    nearest_edge = edges_work.iloc[nearest_idx]

    road_type = nearest_edge.get("highway", "unknown")
    road_name = nearest_edge.get("name", "unknown")
    maxspeed = nearest_edge.get("maxspeed", "unknown")
    lanes = nearest_edge.get("lanes", "unknown")

    if isinstance(road_type, list):
        road_type = road_type[0]
    if isinstance(road_name, list):
        road_name = road_name[0]
    if isinstance(maxspeed, list):
        maxspeed = maxspeed[0]
    if isinstance(lanes, list):
        lanes = lanes[0]

    if pd.isna(road_type):
        road_type = "unknown"
    if pd.isna(road_name):
        road_name = "unknown"
    if pd.isna(maxspeed):
        maxspeed = "unknown"
    if pd.isna(lanes):
        lanes = "unknown"

    eidxs = nearby_edge_indices[i]

    if len(eidxs) > 0:
        road_length_300m = float(edges_work.iloc[eidxs]["length"].sum())
    else:
        road_length_300m = 0.0

    intersection_count_300m = len(nearby_node_indices[i])

    buffer_area = np.pi * (BUFFER_METERS ** 2)
    road_density_300m = road_length_300m / buffer_area

    road_rows.append({
        "grid_code": int(grid_points.iloc[i]["grid_code"]),
        "distance_to_nearest_road_m": float(dist_to_edge[i]),
        "road_length_300m": road_length_300m,
        "intersection_count_300m": int(intersection_count_300m),
        "road_density_300m": float(road_density_300m),
        "nearest_road_type": str(road_type),
        "nearest_road_name": str(road_name),
        "nearest_road_maxspeed": str(maxspeed),
        "nearest_road_lanes": str(lanes)
    })

road_features = pd.DataFrame(road_rows)

print("Road features:", road_features.shape)
display(road_features.head())

ml_data = ml_data.merge(
    road_features,
    on="grid_code",
    how="left"
)

road_numeric_cols = [
    "distance_to_nearest_road_m",
    "road_length_300m",
    "intersection_count_300m",
    "road_density_300m"
]

for col in road_numeric_cols:
    ml_data[col] = pd.to_numeric(ml_data[col], errors="coerce").fillna(0).astype("float32")

road_cat_cols = [
    "nearest_road_type",
    "nearest_road_maxspeed",
    "nearest_road_lanes"
]

for col in road_cat_cols:
    ml_data[col] = ml_data[col].fillna("unknown").astype(str)

print("✅ Road structure and road type features added.")

enriched_path = os.path.join(WORK_DIR, "final_enriched_accident_dataset.parquet")
weather_path = os.path.join(WORK_DIR, "weather_df.parquet")
road_path = os.path.join(WORK_DIR, "road_features.parquet")

ml_data.to_parquet(enriched_path, index=False)
weather_df.to_parquet(weather_path, index=False)
road_features.to_parquet(road_path, index=False)

print("✅ Saved final enriched dataset:", enriched_path)


# ============================================================
# 4. Prepare model data
# ============================================================
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    confusion_matrix
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

cutoff_time = ml_data["datetime_hour"].quantile(0.8)

train_data = ml_data[ml_data["datetime_hour"] < cutoff_time].copy()
test_data = ml_data[ml_data["datetime_hour"] >= cutoff_time].copy()

print("Train:", train_data.shape)
print("Test:", test_data.shape)
print("Cutoff time:", cutoff_time)

numeric_features = [
    "lat_grid",
    "lon_grid",

    "year",
    "month",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_night",

    "past_accidents",
    "past_injuries",
    "past_deaths",

    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "wind_speed_10m",
    "surface_pressure",
    "is_rainy",
    "is_heavy_rain",
    "is_windy",

    "distance_to_nearest_road_m",
    "road_length_300m",
    "intersection_count_300m",
    "road_density_300m"
]

categorical_features = [
    "borough",
    "nearest_road_type",
    "nearest_road_maxspeed",
    "nearest_road_lanes"
]

features = numeric_features + categorical_features

train_data = train_data.dropna(subset=["label"])
test_data = test_data.dropna(subset=["label"])

X_train = train_data[features]
y_train = train_data["label"].astype(int)

X_test = test_data[features]
y_test = test_data["label"].astype(int)


# ============================================================
# 5. Train strong tabular model
# ============================================================
try:
    from lightgbm import LGBMClassifier

    model = LGBMClassifier(
        n_estimators=700,
        learning_rate=0.035,
        num_leaves=63,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_samples=40,
        reg_lambda=1.0,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    print("Using LightGBM model.")

except Exception as e:
    print("LightGBM unavailable. Falling back to HistGradientBoostingClassifier.")
    print("Reason:", e)

    model = HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=42
    )

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", encoder, categorical_features),
        ("num", "passthrough", numeric_features)
    ],
    remainder="drop"
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ]
)

pipeline.fit(X_train, y_train)

print("✅ Model training completed.")


# ============================================================
# 6. Evaluate and tune threshold for best F1
# ============================================================
y_prob = pipeline.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

y_pred_default = (y_prob >= 0.5).astype(int)
y_pred_tuned = (y_prob >= best_threshold).astype(int)

print("\n================ DEFAULT THRESHOLD 0.50 ================")
print(classification_report(y_test, y_pred_default))
print("F1 default:", f1_score(y_test, y_pred_default))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_default))

print("\n================ TUNED THRESHOLD FOR BEST F1 ================")
print("Best threshold:", best_threshold)
print("Best F1:", best_f1)
print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

model_path = os.path.join(WORK_DIR, "best_accident_risk_model.pkl")
features_path = os.path.join(WORK_DIR, "final_model_features.pkl")
threshold_path = os.path.join(WORK_DIR, "best_f1_threshold.pkl")

joblib.dump(pipeline, model_path)
joblib.dump(features, features_path)
joblib.dump(best_threshold, threshold_path)

print("\n✅ Saved model files:")
print(model_path)
print(features_path)
print(threshold_path)


# ============================================================
# 7. Feature importance
# ============================================================
try:
    feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
    importances = pipeline.named_steps["model"].feature_importances_

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    importance_path = os.path.join(WORK_DIR, "feature_importance.csv")
    importance_df.to_csv(importance_path, index=False)

    print("\nTop 20 Feature Importances:")
    display(importance_df.head(20))

except Exception as e:
    print("Feature importance could not be extracted:", e)


# ============================================================
# 8. Predict risk for selected time
# ============================================================
prediction_time = pd.Timestamp("2024-03-22 18:00:00")

base_grid = ml_data[
    ["grid_code", "lat_grid", "lon_grid", "borough"]
].drop_duplicates().copy()

prediction_grid = base_grid.copy()

prediction_grid["year"] = prediction_time.year
prediction_grid["month"] = prediction_time.month
prediction_grid["hour"] = prediction_time.hour
prediction_grid["day_of_week"] = prediction_time.dayofweek
prediction_grid["is_weekend"] = int(prediction_time.dayofweek in [5, 6])
prediction_grid["is_night"] = int((prediction_time.hour >= 20) or (prediction_time.hour <= 5))

history_for_pred = ml_data[
    ["grid_code", "past_accidents", "past_injuries", "past_deaths"]
].drop_duplicates("grid_code")

prediction_grid = prediction_grid.merge(
    history_for_pred,
    on="grid_code",
    how="left"
)

weather_df["datetime_hour"] = pd.to_datetime(weather_df["datetime_hour"])

if prediction_time in set(weather_df["datetime_hour"]):
    weather_row = weather_df[weather_df["datetime_hour"] == prediction_time].iloc[0]
else:
    nearest_idx = (weather_df["datetime_hour"] - prediction_time).abs().idxmin()
    weather_row = weather_df.loc[nearest_idx]
    print("Exact weather time not found. Used nearest weather time:", weather_row["datetime_hour"])

for col in weather_cols:
    prediction_grid[col] = weather_row[col]

prediction_grid["is_rainy"] = (prediction_grid["rain"] > 0).astype(int)
prediction_grid["is_heavy_rain"] = (prediction_grid["rain"] > 2).astype(int)
prediction_grid["is_windy"] = (
    prediction_grid["wind_speed_10m"] > ml_data["wind_speed_10m"].quantile(0.75)
).astype(int)

prediction_grid = prediction_grid.merge(
    road_features,
    on="grid_code",
    how="left"
)

for col in numeric_features:
    prediction_grid[col] = pd.to_numeric(
        prediction_grid[col],
        errors="coerce"
    ).fillna(0)

for col in categorical_features:
    prediction_grid[col] = prediction_grid[col].fillna("unknown").astype(str)

prediction_grid["risk_score"] = pipeline.predict_proba(
    prediction_grid[features]
)[:, 1]

prediction_grid["risk_label"] = (
    prediction_grid["risk_score"] >= best_threshold
).astype(int)

def risk_category(score):
    if score >= 0.80:
        return "Very High Risk"
    elif score >= 0.60:
        return "High Risk"
    elif score >= 0.35:
        return "Medium Risk"
    else:
        return "Low Risk"

prediction_grid["risk_category"] = prediction_grid["risk_score"].apply(risk_category)

def explain_risk(row):
    reasons = []

    if row["past_accidents"] > prediction_grid["past_accidents"].quantile(0.75):
        reasons.append("high past accident history")

    if row["intersection_count_300m"] > prediction_grid["intersection_count_300m"].quantile(0.75):
        reasons.append("many nearby intersections")

    if row["road_density_300m"] > prediction_grid["road_density_300m"].quantile(0.75):
        reasons.append("dense road network")

    if row["rain"] > 0:
        reasons.append("rainy weather")

    if row["wind_speed_10m"] > prediction_grid["wind_speed_10m"].quantile(0.75):
        reasons.append("high wind condition")

    if row["is_night"] == 1:
        reasons.append("night-time risk")

    if len(reasons) == 0:
        reasons.append("risk pattern learned from location, time, weather, and road structure")

    return ", ".join(reasons)

prediction_grid["risk_explanation"] = prediction_grid.apply(explain_risk, axis=1)

prediction_grid = prediction_grid.sort_values(
    "risk_score",
    ascending=False
).reset_index(drop=True)

print("\nTop 20 risky zones:")
display(prediction_grid.head(20))

all_pred_path = os.path.join(WORK_DIR, "final_all_zone_predictions.csv")
top50_path = os.path.join(WORK_DIR, "final_top_50_risky_zones.csv")

prediction_grid.to_csv(all_pred_path, index=False)
prediction_grid.head(50).to_csv(top50_path, index=False)

print("✅ Saved prediction CSV files:")
print(all_pred_path)
print(top50_path)


# ============================================================
# 9. Create final professional map
# ============================================================
import folium
import branca.colormap as cm
from folium.plugins import MiniMap, Fullscreen, MeasureControl

map_df = prediction_grid.head(700).copy()

risk_map = folium.Map(
    location=[map_df["lat_grid"].mean(), map_df["lon_grid"].mean()],
    zoom_start=11,
    tiles="CartoDB positron"
)

colormap = cm.linear.YlOrRd_09.scale(
    map_df["risk_score"].min(),
    map_df["risk_score"].max()
)
colormap.caption = "Predicted Accident Risk Score"

half_step = 0.0005

for _, row in map_df.iterrows():
    lat = row["lat_grid"]
    lon = row["lon_grid"]
    risk = row["risk_score"]
    color = colormap(risk)

    popup_text = f"""
    <div style="font-family: Arial; font-size: 13px;">
    <b>Risk Category:</b> {row['risk_category']}<br>
    <b>Risk Score:</b> {risk:.4f}<br>
    <b>Borough:</b> {row.get('borough', 'UNKNOWN')}<br>
    <b>Past Accidents:</b> {row.get('past_accidents', 0):.0f}<br>
    <b>Rain:</b> {row.get('rain', 0):.2f} mm<br>
    <b>Temperature:</b> {row.get('temperature_2m', 0):.1f} °C<br>
    <b>Wind Speed:</b> {row.get('wind_speed_10m', 0):.1f} km/h<br>
    <b>Nearest Road Type:</b> {row.get('nearest_road_type', 'unknown')}<br>
    <b>Max Speed Tag:</b> {row.get('nearest_road_maxspeed', 'unknown')}<br>
    <b>Lanes Tag:</b> {row.get('nearest_road_lanes', 'unknown')}<br>
    <b>Intersection Count 300m:</b> {row.get('intersection_count_300m', 0):.0f}<br>
    <b>Road Density 300m:</b> {row.get('road_density_300m', 0):.5f}<br>
    <b>Explanation:</b> {row.get('risk_explanation', '')}
    </div>
    """

    folium.Rectangle(
        bounds=[
            [lat - half_step, lon - half_step],
            [lat + half_step, lon + half_step]
        ],
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.72,
        popup=folium.Popup(popup_text, max_width=360),
        tooltip=f"{row['risk_category']} | Score: {risk:.3f}"
    ).add_to(risk_map)

top20 = prediction_grid.head(20).copy()

for rank, (_, row) in enumerate(top20.iterrows(), start=1):
    folium.CircleMarker(
        location=[row["lat_grid"], row["lon_grid"]],
        radius=6,
        color="black",
        weight=2,
        fill=True,
        fill_color="red",
        fill_opacity=1,
        tooltip=f"Top {rank} Risk Zone | Score: {row['risk_score']:.3f}",
        popup=folium.Popup(
            f"<b>Top {rank} Risk Zone</b><br>"
            f"Risk Score: {row['risk_score']:.4f}<br>"
            f"Category: {row['risk_category']}<br>"
            f"Reason: {row['risk_explanation']}",
            max_width=320
        )
    ).add_to(risk_map)

colormap.add_to(risk_map)
MiniMap(toggle_display=True).add_to(risk_map)
Fullscreen().add_to(risk_map)
MeasureControl().add_to(risk_map)

map_path = os.path.join(WORK_DIR, "final_risk_map_weather_road.html")
risk_map.save(map_path)

print("✅ Saved final map:")
print(map_path)

risk_map

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.1 MB/s eta 0:00:00 0:00:01
Loaded ml_data: (3994890, 18)


,grid_code,lat_grid,lon_grid,borough,datetime_hour,accident_count,injured_count,killed_count,label,year,month,hour,day_of_week,is_weekend,is_night,past_accidents,past_injuries,past_deaths
0,38018,40.613998,-74.101997,STATEN ISLAND,2019-08-14 04:00:00,0,0.0,0.0,0,2019,8,4,2,0,1,2.0,0.0,0.0
1,67828,40.728001,-73.738998,UNKNOWN,2022-02-20 11:00:00,0,0.0,0.0,0,2022,2,11,6,1,0,2.0,0.0,0.0
2,48110,40.577999,-73.848000,QUEENS,2019-02-13 21:00:00,0,0.0,0.0,0,2019,2,21,2,0,1,7.0,2.0,0.0
3,33179,40.646999,-74.023003,BROOKLYN,2018-02-06 06:00:00,1,1.0,0.0,1,2018,2,6,1,0,0,84.0,6.0,0.0
4,16307,40.636002,-73.984001,BROOKLYN,2026-01-06 19:00:00,1,1.0,0.0,1,2026,1,19,1,0,0,79.0,21.0,0.0


Fetching historical weather from 2012-07-01 to 2026-04-21
Fetched: 2012-07-01 to 2013-06-30
Fetched: 2013-07-01 to 2014-06-30
Fetched: 2014-07-01 to 2015-06-30
Fetched: 2015-07-01 to 2016-06-30
Fetched: 2016-07-01 to 2017-06-30
Fetched: 2017-07-01 to 2018-06-30
Fetched: 2018-07-01 to 2019-06-30
Fetched: 2019-07-01 to 2020-06-30
Fetched: 2020-07-01 to 2021-06-30
Fetched: 2021-07-01 to 2022-06-30
Fetched: 2022-07-01 to 2023-06-30
Fetched: 2023-07-01 to 2024-06-30
Fetched: 2024-07-01 to 2025-06-30
Fetched: 2025-07-01 to 2026-04-21
Weather data: (121032, 8)


,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,wind_speed_10m,surface_pressure,datetime_hour
0,24.6,46,0.0,0.0,0,9.7,1005.2,2012-07-01 00:00:00
1,23.5,51,0.0,0.0,0,9.7,1005.3,2012-07-01 01:00:00
2,22.6,56,0.0,0.0,0,9.4,1005.2,2012-07-01 02:00:00
3,21.9,62,0.0,0.0,0,8.3,1005.3,2012-07-01 03:00:00
4,21.2,67,0.0,0.0,0,6.8,1005.6,2012-07-01 04:00:00


✅ Weather features added.
left, bottom, right, top = (-74.27499725341796, 40.4790005493164, -73.64300201416016, 40.932998199462894)
OSM nodes: (116332, 8)
OSM edges: (300005, 17)
Road features: (82547, 9)


...

✅ Road structure and road type features added.
✅ Saved final enriched dataset: /kaggle/working/final_enriched_accident_dataset.parquet
Train: (3195879, 36)
Test: (799011, 36)
Cutoff time: 2022-10-26 22:00:00
Using LightGBM model.
[LightGBM] [Info] Number of positive: 1702968, number of negative: 1492911
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.245665 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3007
[LightGBM] [Info] Number of data points in the train set: 3195879, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
✅ Model training completed.

================ DEFAULT THRESHOLD 0.50 ================
              precision    recall  f1-score   support

           0       0.80      0.84      0.82    504534
           

,feature,importance
41,num__lon_grid,4571
40,num__lat_grid,4326
42,num__year,4123
44,num__hour,4050
48,num__past_accidents,3825
51,num__temperature_2m,2059
49,num__past_injuries,1740
62,num__road_length_300m,1723
45,num__day_of_week,1715
61,num__distance_to_nearest_road_m,1534



Top 20 risky zones:


,grid_code,lat_grid,lon_grid,borough,year,month,hour,day_of_week,is_weekend,is_night,...,intersection_count_300m,road_density_300m,nearest_road_type,nearest_road_name,nearest_road_maxspeed,nearest_road_lanes,risk_score,risk_label,risk_category,risk_explanation
0,827,40.692001,-73.999001,UNKNOWN,2024,3,18,4,0,0,...,27,0.016323,motorway,Brooklyn-Queens Expressway,45 mph,2,0.970030,1,Very High Risk,"high past accident history, many nearby inters..."
1,1264,40.820000,-73.890999,BRONX,2024,3,18,4,0,0,...,24,0.015459,secondary,Hunts Point Avenue,unknown,5,0.969862,1,Very High Risk,"high past accident history, many nearby inters..."
2,775,40.703999,-73.817001,UNKNOWN,2024,3,18,4,0,0,...,23,0.021478,primary,Hillside Avenue,25 mph,6,0.968430,1,Very High Risk,"high past accident history, many nearby inters..."
3,751,40.710999,-73.727997,UNKNOWN,2024,3,18,4,0,0,...,24,0.019386,primary,Hempstead Avenue,25 mph,3,0.966779,1,Very High Risk,"high past accident history, many nearby inters..."
4,37,40.862000,-73.913002,BRONX,2024,3,18,4,0,0,...,14,0.021511,primary,West Fordham Road,25 mph,2,0.965136,1,Very High Risk,"high past accident history, dense road network"
5,2046,40.695999,-73.985001,BROOKLYN,2024,3,18,4,0,0,...,26,0.014373,primary,Flatbush Avenue Extension,25 mph,unknown,0.962084,1,Very High Risk,"high past accident history, many nearby inters..."
6,442,40.838001,-73.873001,UNKNOWN,2024,3,18,4,0,0,...,16,0.016428,residential,Bronx River Avenue,25 mph,unknown,0.958919,1,Very High Risk,high past accident history
7,2015,40.675999,-73.897003,BROOKLYN,2024,3,18,4,0,0,...,26,0.016573,primary,Pennsylvania Avenue,unknown,5,0.957550,1,Very High Risk,"high past accident history, many nearby inters..."
8,2031,40.659000,-73.890999,BROOKLYN,2024,3,18,4,0,0,...,25,0.016904,primary,Pennsylvania Avenue,25 mph,unknown,0.957430,1,Very High Risk,"high past accident history, many nearby inters..."
9,248,40.845001,-73.926003,UNKNOWN,2024,3,18,4,0,0,...,17,0.025836,motorway_link,Major Deegan Entrance Northbound,unknown,1,0.956655,1,Very High Risk,"high past accident history, dense road network"


✅ Saved prediction CSV files:
/kaggle/working/final_all_zone_predictions.csv
/kaggle/working/final_top_50_risky_zones.csv
✅ Saved final map:
/kaggle/working/final_risk_map_weather_road.html
